# DNABERT-2 test

To make it work, install all the packages in the `requirements-dnabert.txt` file and run **pip uninstall triton**. If this library is installed, the code will fail. Another way to fix it is to download the `DNABERT-2-117M` repo and in the file `flash_attn_triton.py` change all the *tl.dot(q, k, trans_b=True)* with *tl.dot(q, tl.trans(k))*, and install the latest version available of triton (even if pytorch complains). Please note that with triton installed it will only work on the GPU.

In [1]:
import pandas as pd
import os

## Install dependencies

In [5]:
!git clone https://huggingface.co/zhihan1996/DNABERT-2-117M

Cloning into 'DNABERT-2-117M'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 71 (delta 0), reused 0 (delta 0), pack-reused 68 (from 1)
Unpacking objects: 100% (71/71), 74.94 KiB | 2.20 MiB/s, done.


Now, in the file `flash_attn_triton.py` change all the *tl.dot(q, k, trans_b=True)* with *tl.dot(q, tl.trans(k))*.

In [4]:
%pip install -r requirements-dnabert.txt
%pip uninstall triton -y
%pip install triton

  Using cached triton-2.1.0-0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.3 kB)
Using cached triton-2.1.0-0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (89.2 MB)
Note: you may need to restart the kernel to use updated packages.
Found existing installation: triton 2.1.0
Uninstalling triton-2.1.0:
  Successfully uninstalled triton-2.1.0
Note: you may need to restart the kernel to use updated packages.
  Using cached triton-3.4.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
Using cached triton-3.4.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (155.4 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.1.2 requires triton==2.1.0; platform_system == "Linux" and platform_machine == "x86_64", but you have triton 3.4.0 which is incompatible.
Note: you may need to restart t

## Load data

In [2]:
base_path = "../../data/perphect-data"

couples_df = pd.read_csv(os.path.join(base_path, "public_data_set", "couples_df.csv"))
phages_df = pd.read_csv(os.path.join(base_path, "public_data_set", "phages_df.csv"))
bacteria_df = pd.read_csv(os.path.join(base_path, "public_data_set", "bacteria_df.csv"))
print('couples_df.shape = ', couples_df.shape)
print('phages_df.shape = ', phages_df.shape)
print('bacteria_df.shape = ', bacteria_df.shape)

couples_df.shape =  (4202, 4)
phages_df.shape =  (3459, 3)
bacteria_df.shape =  (94, 3)


In [3]:
phages_df.head()

,phage_id,phage_sequence,sequence_length
0,5326,TGCGGCCGCCCCATCCTGTACGGGTTTCCAAGTCGATCGGAGGGCA...,53124
1,6247,GGCTTTCGTGTGAGCCGTGATGTTTTCACGAATATGTGCCCCACCT...,74483
2,1976,GTGGGAATTTTTTTTTTGGGTTGCGCGGTGATCGCCGATGACGACG...,50781
3,430,ATGGCTTCGACTCAGACTCCAGCCGTCGGCAAGACCACGGCCATCG...,71565
4,431,TGCGGCTGAGCCATCGTGTACGGGTTTCCAAGTCCATCAGAGCCGG...,53396


## Get embedding

In [4]:
import torch
torch.cuda.is_available()

True

In [10]:
import gc
from transformers.models.bert.configuration_bert import BertConfig
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn

In [6]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
def clean_gpu():
    torch.cuda.empty_cache()
    gc.collect()
clean_gpu()
device

'cuda:0'

In [7]:
clean_gpu()
tokenizer = AutoTokenizer.from_pretrained("DNABERT-2-117M/", trust_remote_code=True)
config = BertConfig.from_pretrained("DNABERT-2-117M/")
model = AutoModel.from_pretrained("DNABERT-2-117M/", trust_remote_code=True, config=config)

/home/pere.carrillo/.local/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertModel were not initialized from the model checkpoint at DNABERT-2-117M/ and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
model.to(device)

DataParallel(
  (module): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(4096, 768, padding_idx=0)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertUnpadAttention(
            (self): BertUnpadSelfAttention(
              (dropout): Dropout(p=0.0, inplace=False)
              (Wqkv): Linear(in_features=768, out_features=2304, bias=True)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
          (mlp): BertGatedLinearUnitMLP(
            (gated_layers): Linear(in_features=768, out_f

In [14]:
dna = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"
# dna = phages_df["phage_sequence"].iloc[0]
# dna = bacteria_df["bacterium_sequence"].iloc[0][:171000]
inputs = tokenizer(dna, return_tensors = 'pt')["input_ids"].to(device)
hidden_states = model(inputs)[0] # [1, sequence_length, 768]

# embedding with mean pooling
embedding_mean = torch.mean(hidden_states[0], dim=0)
print(embedding_mean.shape) # expect to be 768

# embedding with max pooling
embedding_max = torch.max(hidden_states[0], dim=0)[0]
print(embedding_max.shape) # expect to be 768

RuntimeError: Caught OutOfResources in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/home/pere.carrillo/.local/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py", line 85, in _worker
    output = module(*input, **kwargs)
  File "/home/pere.carrillo/.local/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1518, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/pere.carrillo/.local/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1527, in _call_impl
    return forward_call(*args, **kwargs)
  File "/home/pere.carrillo/.cache/huggingface/modules/transformers_modules/bert_layers.py", line 609, in forward
    encoder_outputs = self.encoder(
  File "/home/pere.carrillo/.local/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1518, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/pere.carrillo/.local/lib/python3.10/site-packages/torch/nn/modules/modu